In [1]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import pandas as pd
from scipy import stats

from sklearn.model_selection import RandomizedSearchCV 

from sklearn.model_selection import KFold 
from sklearn.model_selection import cross_val_score 
import numpy as np
import sklearn

In [2]:
for i in range(1,18):

    print('')
    print('#_______________ Arquivo ', i, '________________#')
    print('')

    
    
    # Extração dos dados
    
    i=str(i)
    
    X_treino = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_ins_uo.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    
    print('')
    print('Tamanho do treino e quantidade de ocorrências')
    print(len(X_treino.index))
    print(round(sum(X_treino['flg_ocorrencia'])))
    print('')
    
    X_teste = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_oos.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    
    print('')
    print('Tamanho do teste e quantidade de ocorrências')
    print(len(X_teste.index))
    print(round(sum(X_teste['flg_ocorrencia'])))
    print('')
    
    oot = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_final_oot.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    
    print('')
    print('Tamanho do teste 2 meses a frente e quantidade de ocorrências')
    print(len(oot.index))
    print(round(sum(oot['flg_ocorrencia'])))
    print('')

    #Ajuste na flg_ocorrencia
    
    y_treino = X_treino['flg_ocorrencia']
    X_treino = X_treino.drop('flg_ocorrencia', axis=1)
    n = len(X_treino.columns)
    
    y_teste = X_teste['flg_ocorrencia']
    X_teste = X_teste.drop('flg_ocorrencia', axis=1)

    ####################################
    ############ XGboosting ############
    ####################################
    
    classificador_xgboost = XGBClassifier()

    lista_parametros = {'booster': ['gbtree'],
        'max_delta_step': [i + 1 for i in range(1,11,1)],
        'max_depth': [i + 1 for i in range(2,30,1)],
        'eta': stats.uniform(0, 1),
        'objective' : ['reg:logistic', 'binary:logistic', 'binary:logitraw', 'binary:hinge']
        }
 
    rand_search = RandomizedSearchCV(classificador_xgboost, 
                                     param_distributions = lista_parametros,
                                     cv = 10,
                                     random_state = 42,
                                     scoring = 'f1') 

    rand_search.fit(X_treino.iloc[:,3:n], y_treino) 
    
    print('')
    print(rand_search.best_estimator_)
    print('')
    
    classificador_xgboost = rand_search.best_estimator_
    classificador_xgboost.fit(X_treino.iloc[:,3:n], y_treino) 

    
    #############################################
    ############ Qualidade do modelo ############
    #############################################
    
    kfold  = KFold(n_splits=10, shuffle=False)
    result_arvore = cross_val_score(classificador_xgboost, X_treino.iloc[:,3:n], y_treino, cv = kfold)
    #K-fold
    #'Média da acurácia:'
    print(np.mean(result_arvore))
    #'Variância da acurácia'
    print(np.std(result_arvore))

    #Teste - OOS
    y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])),2))

    #Teste - OOT
    y_teste, classificador_xgboost.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(oot.iloc[:,0], classificador_xgboost.predict(oot.iloc[:,4:n+1])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(oot.iloc[:,0], classificador_xgboost.predict(oot.iloc[:,4:n+1])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(oot.iloc[:,0], classificador_xgboost.predict(oot.iloc[:,4:n+1])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(oot.iloc[:,0], classificador_xgboost.predict(oot.iloc[:,4:n+1])),2))


#_______________ Arquivo  1 ________________#


Tamanho do treino e quantidade de ocorrências
6848
3424


Tamanho do teste e quantidade de ocorrências
6900
13


Tamanho do teste 2 meses a frente e quantidade de ocorrências
135288
230


XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eta=0.9922115592912175, eval_metric=None, feature_types=None,
              gamma=0, gpu_id=-1, grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.99221158, max_bin=256,
              max_cat_threshold=64, max_cat_to_onehot=4, max_delta_step=2,
              max_depth=14, max_leaves=0, min_child_weight=1, missing=nan,
              monotone_constraints='()', n_estimators=100, n_jobs=0,
              num_parallel_tree=1, predictor='auto', ...)

0.9960583941605841
0.005949688721843757


0.9978414827739261
0.005376285692991672
1.0
0.47
0.33
0.39
0.95
0.0
0.0
0.0

#_______________ Arquivo  9 ________________#


Tamanho do treino e quantidade de ocorrências
120728
60364


Tamanho do teste e quantidade de ocorrências
121141
158


Tamanho do teste 2 meses a frente e quantidade de ocorrências
4907
17


XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eta=0.3663618432936917, eval_metric=None, feature_types=None,
              gamma=0, gpu_id=-1, grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.366361856,
              max_bin=256, max_cat_threshold=64, max_cat_to_onehot=4,
              max_delta_step=4, max_depth=14, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=100,
              n_jobs=0, num_p

KeyboardInterrupt: 